In [ ]:
import json

import pandas as pd
import requests
from pydantic import BaseModel, ConfigDict, Field, ValidationError

In [ ]:
corals_data_url = (
    r"https://www.ncei.noaa.gov/erddap/tabledap/deep_sea_corals.jsonlKVP"
    r"?CatalogNumber,RecordType,LocationAccuracy,ScientificName,latitude,longitude,DepthInMeters,LargeMarineEcosystem,ObservationDate,EventID,SurveyID,AphiaID,Genus,Family,Order,IndividualCount,Density,Temperature,Salinity,Oxygen,SampleAreaInSquareMeters,gisGEBCODepth,Country,Substrate,CMECSSubstrate,CMECSGeoForm,ShallowFlag"
    r"&ShallowFlag=0"
    r"&ObservationYear%3E=2018"
    r"&Country=%22USA%22"
    r"&LargeMarineEcosystem=~%22%28Southeast%20U.S.%20Continental%20Shelf%7CGulf%20of%20Mexico%7CCaribbean%20Sea%29%22"
)

In [ ]:
class DeepSeaCoralRecord(BaseModel):
    model_config = ConfigDict(populate_by_name=True, frozen=True)

    lat: float = Field(alias="latitude")
    lon: float = Field(alias="longitude")
    catalog_num: int = Field(alias="CatalogNumber")
    record_type: str = Field(alias="RecordType")
    loc_accuracy: str | None = Field(default=None, alias="LocationAccuracy")
    scientific_nm: str | None = Field(default=None, alias="ScientificName")
    depth: float | None = Field(default=None, alias="DepthInMeters")
    gis_depth: float | None = Field(default=None, alias="gisGEBCODepth")
    large_marine_ecosystem: str = Field(alias="LargeMarineEcosystem")
    country: str | None = Field(default=None, alias="Country")
    obs_date: str = Field(alias="ObservationDate")
    event_id: str = Field(alias="EventID")
    survey_id: str = Field(alias="SurveyID")
    aphia_id: int | None = Field(default=None, alias="AphiaID")
    genus: str | None = Field(default=None, alias="Genus")
    family: str | None = Field(default=None, alias="Family")
    order: str | None = Field(default=None, alias="Order")
    count: int | None = Field(default=None, alias="IndividualCount")
    density: float | None = Field(default=None, alias="Density")
    sample_area: float | None = Field(default=None, alias="SampleAreaInSquareMeters")
    temp: float | None = Field(default=None, alias="Temperature")
    salinity: float | None = Field(default=None, alias="Salinity")
    oxygen: float | None = Field(default=None, alias="Oxygen")
    substrate: str | None = Field(default=None, alias="Substrate")
    cmecs_substrate: str | None = Field(default=None, alias="CMECSSubstrate")
    cmecs_geo_form: str | None = Field(default=None, alias="CMECSGeoForm")
    is_shallow: bool | None = Field(default=None, alias="ShallowFlag")


In [ ]:
def download_coral(url: str) -> pd.DataFrame:
    records: list[DeepSeaCoralRecord] = []

    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            record = DeepSeaCoralRecord(**json.loads(line))
            records.append(record)

    return pd.DataFrame(record.model_dump() for record in records)

In [ ]:
df = download_coral(corals_data_url)
df.head()